# Logistic Regression: Complete Mathematical Guide & Implementation

## Learning Objectives
By the end of this notebook, you will understand:
- Mathematical foundations of logistic regression with complete derivations
- Maximum likelihood estimation and optimization techniques
- Iteratively Reweighted Least Squares (IRLS) algorithm
- Multinomial logistic regression for multi-class classification
- Regularization techniques (Ridge and Lasso) for logistic regression  
- Model evaluation, diagnostics, and interpretation
- Comparison with linear regression and practical applications

## Table of Contents
1. Mathematical Foundation & Theory
2. Binary Logistic Regression Implementation
3. Optimization Algorithms (Gradient Descent, Newton-Raphson, IRLS)
4. Multinomial Logistic Regression
5. Regularization Techniques
6. Model Evaluation & Diagnostics
7. Practical Implementation & Comparison
8. Real-World Applications

---

## Mathematical Foundation & Theory

### What is Logistic Regression?

Logistic regression is a **statistical method** for **binary and multinomial classification**. Unlike linear regression which predicts continuous values, logistic regression predicts the **probability** that an instance belongs to a particular class.

### 1. The Sigmoid (Logistic) Function

The core of logistic regression is the **sigmoid function** (also called the **logistic function**):

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Properties of the Sigmoid Function:**
- **Domain**: $z \in (-\infty, +\infty)$
- **Range**: $\sigma(z) \in (0, 1)$
- **Monotonic**: Strictly increasing function
- **Symmetric**: $\sigma(-z) = 1 - \sigma(z)$
- **Derivative**: $\frac{d\sigma(z)}{dz} = \sigma(z)(1 - \sigma(z))$

### 2. Linear Combination & Logit

The input to the sigmoid function is a **linear combination** of features:

$$z = \boldsymbol{\beta}^T\mathbf{x} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n$$

Where:
- $\mathbf{x} = [1, x_1, x_2, \ldots, x_n]^T$ is the feature vector (with bias term)
- $\boldsymbol{\beta} = [\beta_0, \beta_1, \ldots, \beta_n]^T$ is the parameter vector
- $\beta_0$ is the **intercept** (bias term)
- $\beta_1, \ldots, \beta_n$ are the **coefficients** for each feature

### 3. Probability Estimation

The **predicted probability** that an instance belongs to class 1 is:

$$P(y = 1 | \mathbf{x}; \boldsymbol{\beta}) = \sigma(\boldsymbol{\beta}^T\mathbf{x}) = \frac{1}{1 + e^{-\boldsymbol{\beta}^T\mathbf{x}}}$$

Similarly, the probability for class 0 is:

$$P(y = 0 | \mathbf{x}; \boldsymbol{\beta}) = 1 - \sigma(\boldsymbol{\beta}^T\mathbf{x}) = \frac{e^{-\boldsymbol{\beta}^T\mathbf{x}}}{1 + e^{-\boldsymbol{\beta}^T\mathbf{x}}}$$

### 4. Odds and Log-Odds (Logit)

**Odds** represent the ratio of the probability of success to failure:

$$\text{Odds} = \frac{P(y = 1 | \mathbf{x})}{P(y = 0 | \mathbf{x})} = \frac{P(y = 1 | \mathbf{x})}{1 - P(y = 1 | \mathbf{x})}$$

**Log-odds** (or **logit**) is the natural logarithm of the odds:

$$\text{logit}(p) = \ln\left(\frac{p}{1-p}\right) = \boldsymbol{\beta}^T\mathbf{x}$$

**Key Insight**: The logit transformation creates a **linear relationship** between the features and the log-odds, making the model interpretable!

### 5. Maximum Likelihood Estimation (MLE)

#### Likelihood Function

For $m$ training examples $\{(\mathbf{x}_i, y_i)\}_{i=1}^m$ where $y_i \in \{0, 1\}$:

The **likelihood function** is the probability of observing the data given the parameters:

$$L(\boldsymbol{\beta}) = \prod_{i=1}^{m} P(y_i | \mathbf{x}_i; \boldsymbol{\beta})$$

Since $P(y_i | \mathbf{x}_i; \boldsymbol{\beta}) = p_i^{y_i}(1-p_i)^{1-y_i}$ where $p_i = \sigma(\boldsymbol{\beta}^T\mathbf{x}_i)$:

$$L(\boldsymbol{\beta}) = \prod_{i=1}^{m} p_i^{y_i}(1-p_i)^{1-y_i}$$

#### Log-Likelihood Function

Taking the natural logarithm (for easier optimization):

$$\ell(\boldsymbol{\beta}) = \ln L(\boldsymbol{\beta}) = \sum_{i=1}^{m} \left[ y_i \ln(p_i) + (1-y_i) \ln(1-p_i) \right]$$

**Alternative formulation** using the sigmoid properties:
$$\ell(\boldsymbol{\beta}) = \sum_{i=1}^{m} \left[ y_i \boldsymbol{\beta}^T\mathbf{x}_i - \ln(1 + e^{\boldsymbol{\beta}^T\mathbf{x}_i}) \right]$$

#### Cost Function

We typically **minimize the negative log-likelihood** (cross-entropy loss):

$$J(\boldsymbol{\beta}) = -\ell(\boldsymbol{\beta}) = -\sum_{i=1}^{m} \left[ y_i \ln(p_i) + (1-y_i) \ln(1-p_i) \right]$$

### 6. Gradient and Hessian

#### Gradient Vector

The **gradient** of the log-likelihood with respect to $\boldsymbol{\beta}$:

$$\nabla_{\boldsymbol{\beta}} \ell(\boldsymbol{\beta}) = \sum_{i=1}^{m} (y_i - p_i) \mathbf{x}_i = \mathbf{X}^T(\mathbf{y} - \mathbf{p})$$

Where:
- $\mathbf{X} = [\mathbf{x}_1, \mathbf{x}_2, \ldots, \mathbf{x}_m]^T$ is the $m \times (n+1)$ design matrix
- $\mathbf{y} = [y_1, y_2, \ldots, y_m]^T$ is the target vector
- $\mathbf{p} = [p_1, p_2, \ldots, p_m]^T$ is the predicted probability vector

#### Hessian Matrix

The **Hessian matrix** (second derivatives):

$$\mathbf{H} = \nabla^2_{\boldsymbol{\beta}} \ell(\boldsymbol{\beta}) = -\sum_{i=1}^{m} p_i(1-p_i) \mathbf{x}_i \mathbf{x}_i^T = -\mathbf{X}^T\mathbf{W}\mathbf{X}$$

Where $\mathbf{W}$ is a diagonal matrix with $W_{ii} = p_i(1-p_i)$.

**Important Property**: The Hessian is **negative definite**, making the log-likelihood function **concave**, which guarantees a unique global maximum!

### 7. Why No Closed-Form Solution?

Unlike linear regression, logistic regression has **no closed-form solution** because:

1. The sigmoid function introduces **non-linearity**
2. The log-likelihood involves **transcendental functions** (exponentials and logarithms)  
3. Setting $\nabla_{\boldsymbol{\beta}} \ell(\boldsymbol{\beta}) = \mathbf{0}$ leads to **nonlinear equations**

Therefore, we need **iterative optimization algorithms**!

---

## Multinomial Logistic Regression (Softmax Regression)

### Mathematical Foundation

For **multi-class classification** with $K$ classes, we extend binary logistic regression using the **softmax function**.

#### Softmax Function

For class $k$ out of $K$ classes:
$$P(y = k | \mathbf{x}; \boldsymbol{\beta}) = \frac{e^{\boldsymbol{\beta}_k^T\mathbf{x}}}{\sum_{j=1}^K e^{\boldsymbol{\beta}_j^T\mathbf{x}}}$$

**Properties:**
- $P(y = k | \mathbf{x}) \in (0, 1)$ for all $k$
- $\sum_{k=1}^K P(y = k | \mathbf{x}) = 1$ (probabilities sum to 1)
- **Generalization**: When $K = 2$, softmax reduces to binary logistic regression

#### Log-Likelihood for Multinomial Case

$$\ell(\boldsymbol{\beta}) = \sum_{i=1}^{m} \sum_{k=1}^K y_{ik} \ln P(y = k | \mathbf{x}_i; \boldsymbol{\beta})$$

Where $y_{ik} = 1$ if sample $i$ belongs to class $k$, and $0$ otherwise (one-hot encoding).

#### Gradient

For parameter vector $\boldsymbol{\beta}_k$ (class $k$):
$$\nabla_{\boldsymbol{\beta}_k} \ell(\boldsymbol{\beta}) = \sum_{i=1}^{m} (y_{ik} - P(y = k | \mathbf{x}_i)) \mathbf{x}_i$$

---

In [ ]:
# Implementation

def softmax(Z):
    """
    Compute softmax function with numerical stability
    Z: (m, K) matrix where each row is the linear combination for each class
    """
    # Subtract max for numerical stability
    Z_stable = Z - np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(Z_stable)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

def one_hot_encode(y, n_classes):
    """Convert labels to one-hot encoding"""
    Y = np.zeros((len(y), n_classes))
    Y[np.arange(len(y)), y] = 1
    return Y

class MultinomialLogisticRegression:
    def __init__(self, max_iter=1000, tol=1e-6, learning_rate=0.01, regularization=0.0):
        self.max_iter = max_iter
        self.tol = tol
        self.learning_rate = learning_rate
        self.regularization = regularization
        self.beta = None
        self.n_classes = None
        self.cost_history = []
    
    def fit(self, X, y):
        """
        Fit multinomial logistic regression using gradient descent
        """
        m, n = X.shape
        self.n_classes = len(np.unique(y))
        
        # Initialize parameters: (n_features, n_classes)
        self.beta = np.random.normal(0, 0.01, (n, self.n_classes))
        
        # Convert labels to one-hot
        Y = one_hot_encode(y, self.n_classes)
        
        for i in range(self.max_iter):
            # Forward pass
            Z = X @ self.beta  # (m, K)
            P = softmax(Z)     # (m, K)
            
            # Compute cost (negative log-likelihood + regularization)
            log_likelihood = np.sum(Y * np.log(P + 1e-15))
            regularization_term = self.regularization * np.sum(self.beta**2)
            cost = -log_likelihood + regularization_term
            self.cost_history.append(cost)
            
            # Compute gradient
            gradient = X.T @ (P - Y) + 2 * self.regularization * self.beta
            
            # Update parameters
            beta_new = self.beta - self.learning_rate * gradient
            
            # Check convergence
            if np.linalg.norm(beta_new - self.beta) < self.tol:
                print(f"   • Converged at iteration {i+1}")
                break
                
            self.beta = beta_new
        
        return self
    
    def predict_proba(self, X):
        """Predict class probabilities"""
        Z = X @ self.beta
        return softmax(Z)
    
    def predict(self, X):
        """Predict classes"""
        probabilities = self.predict_proba(X)
        return np.argmax(probabilities, axis=1)

def multinomial_irls(X, y, max_iter=100, tol=1e-6, regularization=0.0):
    """
    Multinomial Logistic Regression using IRLS
    """
    m, n = X.shape
    n_classes = len(np.unique(y))
    
    # Initialize parameters
    beta = np.random.normal(0, 0.01, (n, n_classes))
    Y = one_hot_encode(y, n_classes)
    
    for iteration in range(max_iter):
        # Forward pass
        Z = X @ beta
        P = softmax(Z)
        
        beta_old = beta.copy()
        
        # Update each class separately (coordinate descent style)
        for k in range(n_classes):
            # Working response and weights for class k
            p_k = P[:, k]
            y_k = Y[:, k]
            
            # Weights (diagonal of Fisher information matrix)
            w_k = p_k * (1 - p_k)
            W_k = np.diag(w_k + 1e-8)  # Add small constant for stability
            
            # Working response
            z_k = Z[:, k] + (y_k - p_k) / (w_k + 1e-8)
            
            # Weighted least squares update
            XTW = X.T @ W_k
            A = XTW @ X + regularization * np.eye(n)
            b = XTW @ z_k
            
            try:
                beta[:, k] = np.linalg.solve(A, b)
            except np.linalg.LinAlgError:
                beta[:, k] = np.linalg.pinv(A) @ b
        
        # Check convergence
        if np.linalg.norm(beta - beta_old) < tol:
            print(f"   • Converged at iteration {iteration+1}")
            break
    
    return beta

# Test on Iris Dataset (3-class classification)

print("\n🌸 MULTINOMIAL LOGISTIC REGRESSION - IRIS DATASET")
print("=" * 55)

# Load Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

print(f"Dataset: {X_iris.shape[0]} samples, {X_iris.shape[1]} features")
print(f"Classes: {iris.target_names}")
print(f"Class distribution: {np.bincount(y_iris)}")

# Split data
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

# Standardize features
scaler_iris = StandardScaler()
X_train_iris_scaled = scaler_iris.fit_transform(X_train_iris)
X_test_iris_scaled = scaler_iris.transform(X_test_iris)

# Add bias term
X_train_iris_bias = np.hstack([np.ones((X_train_iris_scaled.shape[0], 1)), X_train_iris_scaled])
X_test_iris_bias = np.hstack([np.ones((X_test_iris_scaled.shape[0], 1)), X_test_iris_scaled])

print(f"\n🔧 Training Methods:")

# Method 1: Gradient Descent
print("\n1. Gradient Descent:")
multinomial_gd = MultinomialLogisticRegression(max_iter=2000, learning_rate=0.1, regularization=0.01)
multinomial_gd.fit(X_train_iris_bias, y_train_iris)

y_pred_gd = multinomial_gd.predict(X_test_iris_bias)
y_prob_gd = multinomial_gd.predict_proba(X_test_iris_bias)
accuracy_gd = accuracy_score(y_test_iris, y_pred_gd)
print(f"   • Test Accuracy: {accuracy_gd:.4f}")

# Method 2: IRLS
print("\n2. IRLS:")
beta_irls = multinomial_irls(X_train_iris_bias, y_train_iris, regularization=0.01)

# Predictions using IRLS
Z_test_irls = X_test_iris_bias @ beta_irls
y_prob_irls = softmax(Z_test_irls)
y_pred_irls = np.argmax(y_prob_irls, axis=1)
accuracy_irls = accuracy_score(y_test_iris, y_pred_irls)
print(f"   • Test Accuracy: {accuracy_irls:.4f}")

# Method 3: Scikit-learn comparison
print("\n3. Scikit-learn (Reference):")
sklearn_multi = SKLogisticRegression(max_iter=1000, C=100, random_state=42)  # C=1/regularization
sklearn_multi.fit(X_train_iris_scaled, y_train_iris)
y_pred_sklearn = sklearn_multi.predict(X_test_iris_scaled)
accuracy_sklearn = accuracy_score(y_test_iris, y_pred_sklearn)
print(f"   • Test Accuracy: {accuracy_sklearn:.4f}")

# Confusion matrices
print("\n📊 CONFUSION MATRICES")
print("-" * 25)

methods = ['Gradient Descent', 'IRLS', 'Scikit-learn']
predictions = [y_pred_gd, y_pred_irls, y_pred_sklearn]

for method, pred in zip(methods, predictions):
    print(f"\n{method}:")
    cm = confusion_matrix(y_test_iris, pred)
    print(cm)

# Parameter comparison
print("\n🎯 PARAMETER COMPARISON")
print("-" * 25)
feature_names_iris = ['Intercept'] + list(iris.feature_names)

print("\nGradient Descent Parameters:")
print(f"{'Feature':<20} {'Setosa':<10} {'Versicolor':<12} {'Virginica':<10}")
print("-" * 55)
for i, feature in enumerate(feature_names_iris):
    params = multinomial_gd.beta[i, :]
    print(f"{feature:<20} {params[0]:<10.4f} {params[1]:<12.4f} {params[2]:<10.4f}")

print("\nIRLS Parameters:")
print(f"{'Feature':<20} {'Setosa':<10} {'Versicolor':<12} {'Virginica':<10}")
print("-" * 55)
for i, feature in enumerate(feature_names_iris):
    params = beta_irls[i, :]
    print(f"{feature:<20} {params[0]:<10.4f} {params[1]:<12.4f} {params[2]:<10.4f}")

In [ ]:
# Essential libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression as SKLogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                           confusion_matrix, classification_report, roc_auc_score, roc_curve)
import warnings
import time
from scipy import stats
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

# Configuration for better visualizations
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
sns.set_palette("husl")

print("🚀 Logistic Regression Comprehensive Analysis")
print("=" * 60)
print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")
print("=" * 60)

## Binary Logistic Regression Implementation

## Optimization Algorithms for Logistic Regression

Since logistic regression has no closed-form solution, we need iterative optimization algorithms. Let's implement and compare different approaches.

### Mathematical Foundation: Optimization Theory

For logistic regression, we want to find:
$$\hat{\boldsymbol{\beta}} = \arg\max_{\boldsymbol{\beta}} \ell(\boldsymbol{\beta}) = \arg\min_{\boldsymbol{\beta}} J(\boldsymbol{\beta})$$

Where $J(\boldsymbol{\beta}) = -\ell(\boldsymbol{\beta})$ is the **cross-entropy loss**.

---

## Optimization Method 1: Gradient Descent

### Mathematical Derivation

**Gradient Descent Update Rule:**
$$\boldsymbol{\beta}^{(t+1)} = \boldsymbol{\beta}^{(t)} + \alpha \nabla_{\boldsymbol{\beta}} \ell(\boldsymbol{\beta}^{(t)})$$

Where $\alpha > 0$ is the **learning rate** and:
$$\nabla_{\boldsymbol{\beta}} \ell(\boldsymbol{\beta}) = \mathbf{X}^T(\mathbf{y} - \mathbf{p})$$

---

## Optimization Method 2: Newton-Raphson Method

### Mathematical Derivation

**Newton-Raphson Update Rule:**
$$\boldsymbol{\beta}^{(t+1)} = \boldsymbol{\beta}^{(t)} - \mathbf{H}^{-1}(\boldsymbol{\beta}^{(t)}) \nabla_{\boldsymbol{\beta}} J(\boldsymbol{\beta}^{(t)})$$

For **maximization** (log-likelihood):
$$\boldsymbol{\beta}^{(t+1)} = \boldsymbol{\beta}^{(t)} - \mathbf{H}^{-1}(\boldsymbol{\beta}^{(t)}) \nabla_{\boldsymbol{\beta}} \ell(\boldsymbol{\beta}^{(t)})$$

Where the **Hessian** is: $\mathbf{H} = -\mathbf{X}^T\mathbf{W}\mathbf{X}$ and $W_{ii} = p_i(1-p_i)$

---

## Optimization Method 3: Iteratively Reweighted Least Squares (IRLS)

### Mathematical Derivation

IRLS is a **special case of Newton-Raphson** that reformulates logistic regression as a sequence of **weighted least squares** problems.

**Key Insight**: At each iteration, solve:
$$\boldsymbol{\beta}^{(t+1)} = \arg\min_{\boldsymbol{\beta}} (\mathbf{z}^{(t)} - \mathbf{X}\boldsymbol{\beta})^T\mathbf{W}^{(t)}(\mathbf{z}^{(t)} - \mathbf{X}\boldsymbol{\beta})$$

Where:
- **Adjusted response**: $z_i^{(t)} = \boldsymbol{\beta}^{(t)T}\mathbf{x}_i + \frac{y_i - p_i^{(t)}}{p_i^{(t)}(1-p_i^{(t)})}$
- **Weights**: $W_{ii}^{(t)} = p_i^{(t)}(1-p_i^{(t)})$

**IRLS Update Formula:**
$$\boldsymbol{\beta}^{(t+1)} = (\mathbf{X}^T\mathbf{W}^{(t)}\mathbf{X})^{-1}\mathbf{X}^T\mathbf{W}^{(t)}\mathbf{z}^{(t)}$$

---

In [ ]:
# Implementation

def sigmoid(z):
    """
    Compute sigmoid function with numerical stability
    """
    # Clip z to prevent overflow
    z = np.clip(z, -250, 250)
    return 1 / (1 + np.exp(-z))

def logistic_regression_gradient_descent(X, y, learning_rate=0.01, max_iter=1000, tol=1e-6):
    """
    Logistic Regression using Gradient Descent
    """
    m, n = X.shape
    beta = np.zeros(n)
    cost_history = []
    
    for i in range(max_iter):
        # Forward pass
        z = X @ beta
        p = sigmoid(z)
        
        # Compute cost (negative log-likelihood)
        cost = -np.sum(y * np.log(p + 1e-15) + (1 - y) * np.log(1 - p + 1e-15)) / m
        cost_history.append(cost)
        
        # Compute gradient
        gradient = X.T @ (p - y) / m
        
        # Update parameters
        beta_new = beta - learning_rate * gradient
        
        # Check convergence
        if np.linalg.norm(beta_new - beta) < tol:
            print(f"   • Converged at iteration {i+1}")
            break
            
        beta = beta_new
    
    return beta, cost_history

def logistic_regression_newton_raphson(X, y, max_iter=100, tol=1e-6):
    """
    Logistic Regression using Newton-Raphson Method
    """
    m, n = X.shape
    beta = np.zeros(n)
    
    for i in range(max_iter):
        # Forward pass
        z = X @ beta
        p = sigmoid(z)
        
        # Compute weights for Hessian
        w = p * (1 - p)
        W = np.diag(w)
        
        # Compute gradient and Hessian
        gradient = X.T @ (y - p)  # For maximization
        hessian = -X.T @ W @ X    # Negative definite
        
        # Newton-Raphson update
        try:
            beta_new = beta - np.linalg.solve(hessian, gradient)
        except np.linalg.LinAlgError:
            # Use pseudo-inverse if matrix is singular
            beta_new = beta - np.linalg.pinv(hessian) @ gradient
        
        # Check convergence
        if np.linalg.norm(beta_new - beta) < tol:
            print(f"   • Converged at iteration {i+1}")
            break
            
        beta = beta_new
    
    return beta

def logistic_regression_irls(X, y, max_iter=100, tol=1e-6, regularization=0.0):
    """
    Logistic Regression using Iteratively Reweighted Least Squares (IRLS)
    """
    m, n = X.shape
    beta = np.zeros(n)
    
    for i in range(max_iter):
        # Forward pass
        z_linear = X @ beta
        p = sigmoid(z_linear)
        
        # Avoid numerical issues
        p = np.clip(p, 1e-15, 1 - 1e-15)
        
        # Compute weights
        w = p * (1 - p)
        W = np.diag(w)
        
        # Compute adjusted response (working response)
        z_adjusted = z_linear + (y - p) / w
        
        # Solve weighted least squares with regularization
        XTW = X.T @ W
        A = XTW @ X + regularization * np.eye(n)
        b = XTW @ z_adjusted
        
        try:
            beta_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            beta_new = np.linalg.pinv(A) @ b
        
        # Check convergence
        if np.linalg.norm(beta_new - beta) < tol:
            print(f"   • Converged at iteration {i+1}")
            break
            
        beta = beta_new
    
    return beta

def predict_probabilities(X, beta):
    """Predict class probabilities"""
    return sigmoid(X @ beta)

def predict_classes(X, beta, threshold=0.5):
    """Predict binary classes"""
    probabilities = predict_probabilities(X, beta)
    return (probabilities >= threshold).astype(int)

def compute_metrics(y_true, y_pred, y_prob):
    """Compute comprehensive evaluation metrics"""
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'auc_roc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan
    }

# Load and Prepare Dataset for Binary Classification

# Load breast cancer dataset (binary classification)
print("📊 Loading Breast Cancer Dataset")
print("-" * 35)
cancer = load_breast_cancer()
X_full = cancer.data
y_full = cancer.target

print(f"Dataset shape: {X_full.shape}")
print(f"Classes: {cancer.target_names}")
print(f"Features: {len(cancer.feature_names)} total")
print(f"Class distribution: {np.bincount(y_full)}")

# Use subset of features for interpretability
feature_indices = [0, 1, 2, 3, 6, 7]  # Select most important features
feature_names = [cancer.feature_names[i] for i in feature_indices]
X = X_full[:, feature_indices]

print(f"Selected features: {feature_names}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_full, test_size=0.2, random_state=42, stratify=y_full
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Add bias term
X_train_bias = np.hstack([np.ones((X_train_scaled.shape[0], 1)), X_train_scaled])
X_test_bias = np.hstack([np.ones((X_test_scaled.shape[0], 1)), X_test_scaled])

print(f"\n📋 Data Preparation Complete:")
print(f"   • Training set: {X_train_bias.shape[0]} samples")
print(f"   • Test set: {X_test_bias.shape[0]} samples")
print(f"   • Features (with bias): {X_train_bias.shape[1]}")

# Compare Optimization Methods

print("\n🔍 COMPARING OPTIMIZATION METHODS")
print("=" * 45)

methods = {
    'Gradient Descent': lambda X, y: logistic_regression_gradient_descent(X, y, learning_rate=0.01, max_iter=1000),
    'Newton-Raphson': lambda X, y: logistic_regression_newton_raphson(X, y, max_iter=100),
    'IRLS': lambda X, y: logistic_regression_irls(X, y, max_iter=100),
}

results = {}

for method_name, method_func in methods.items():
    print(f"\n🔧 Training with {method_name}:")
    start_time = time.time()
    
    if method_name == 'Gradient Descent':
        beta, cost_history = method_func(X_train_bias, y_train)
        results[method_name] = {'beta': beta, 'cost_history': cost_history}
    else:
        beta = method_func(X_train_bias, y_train)
        results[method_name] = {'beta': beta, 'cost_history': None}
    
    training_time = time.time() - start_time
    
    # Make predictions
    y_train_prob = predict_probabilities(X_train_bias, beta)
    y_test_prob = predict_probabilities(X_test_bias, beta)
    y_train_pred = predict_classes(X_train_bias, beta)
    y_test_pred = predict_classes(X_test_bias, beta)
    
    # Compute metrics
    train_metrics = compute_metrics(y_train, y_train_pred, y_train_prob)
    test_metrics = compute_metrics(y_test, y_test_pred, y_test_prob)
    
    results[method_name].update({
        'train_metrics': train_metrics,
        'test_metrics': test_metrics,
        'training_time': training_time
    })
    
    print(f"   • Training time: {training_time:.4f} seconds")
    print(f"   • Train accuracy: {train_metrics['accuracy']:.4f}")
    print(f"   • Test accuracy: {test_metrics['accuracy']:.4f}")
    print(f"   • Test AUC-ROC: {test_metrics['auc_roc']:.4f}")

# Compare parameter estimates
print(f"\n📊 PARAMETER COMPARISON")
print("-" * 25)
feature_names_full = ['Intercept'] + feature_names
print(f"{'Feature':<20} {'Grad Descent':<12} {'Newton-Raph':<12} {'IRLS':<12}")
print("-" * 60)

for i, feature in enumerate(feature_names_full):
    gd_coef = results['Gradient Descent']['beta'][i]
    nr_coef = results['Newton-Raphson']['beta'][i]
    irls_coef = results['IRLS']['beta'][i]
    print(f"{feature:<20} {gd_coef:<12.4f} {nr_coef:<12.4f} {irls_coef:<12.4f}")

# Check parameter similarity
gd_beta = results['Gradient Descent']['beta']
nr_beta = results['Newton-Raphson']['beta']
irls_beta = results['IRLS']['beta']

print(f"\n🎯 Parameter Similarity:")
print(f"   • GD vs Newton-Raphson: {np.linalg.norm(gd_beta - nr_beta):.6f}")
print(f"   • GD vs IRLS: {np.linalg.norm(gd_beta - irls_beta):.6f}")
print(f"   • Newton-Raphson vs IRLS: {np.linalg.norm(nr_beta - irls_beta):.6f}")